# Notebook 05 (v5.5) — Measurement Pipeline: Theory‑Aligned Composites & Indices

**Purpose:** Build *reproducible* theory‑aligned composites and indices from BERTopic outputs, with transparent auditing and measurement diagnostics.

**This notebook does NOT test hypotheses.** It produces:
- topic membership tables (per composite)
- pipeline audit tables (counts per stage)
- composite quality labels (CORE vs EXPLORATORY; UNIDIMENSIONAL vs MULTIDIMENSIONAL; stability metrics)
- book‑level indices (raw + z)
- segment‑level indices (begin/middle/end)
- derived indices (log‑ratios / contrasts), outcome‑blind

Downstream hypothesis testing happens in **Notebook 06**.


## Section 0 — Configuration (portable paths + thresholds)

In [1]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional, Dict, List, Any, Tuple
import os
import json
import datetime

import numpy as np
import pandas as pd

# ------------------------------
# Portable project root
# ------------------------------
# Preferred: set ROMANCE_ROOT=/path/to/project
# Fallback: infer from current working directory
if "ROMANCE_ROOT" in os.environ:
    PROJECT_ROOT = Path(os.environ.get("ROMANCE_ROOT")).resolve()
else:
    # Try to find project root by looking for markers (results/, notebooks/, etc.)
    cwd = Path.cwd().resolve()
    # If we're in a subdirectory, navigate up to find project root
    # Project root should have both 'results' and 'notebooks' directories
    candidate = cwd
    max_depth = 10
    depth = 0
    while depth < max_depth:
        if (candidate / "results").exists() and (candidate / "notebooks").exists():
            PROJECT_ROOT = candidate
            break
        if candidate.parent == candidate:  # Reached filesystem root
            PROJECT_ROOT = cwd  # Fallback to current directory
            break
        candidate = candidate.parent
        depth += 1
    else:
        PROJECT_ROOT = cwd  # Fallback to current directory

# Correlation analysis results directory
CORRELATION_ANALYSIS_DIR = PROJECT_ROOT / "results" / "correlation_analysis"
RESULTS_DIR = PROJECT_ROOT / "results" / "measurement_v5"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

@dataclass(frozen=True)
class PipelineConfig:
    # Input files (parquet strongly recommended)
    # Files are located in results/correlation_analysis/ subdirectories
    topic_lookup_path: Path = CORRELATION_ANALYSIS_DIR / "data_preparation" / "taxonomy_radway_eda" / "topic_lookup.parquet"
    book_topic_probs_path: Path = CORRELATION_ANALYSIS_DIR / "data_preparation" / "topic_probabilities" / "book_topic_probs.parquet"
    segment_topic_probs_path: Optional[Path] = CORRELATION_ANALYSIS_DIR / "data_preparation" / "topic_probabilities" / "tertile_topic_probs.parquet"  # optional: begin/middle/end segments

    # Optional diagnostics
    topic_health_path: Optional[Path] = CORRELATION_ANALYSIS_DIR / "01_topic_analysis" / "tables" / "topic_health_table.parquet"
    author_dominance_path: Optional[Path] = CORRELATION_ANALYSIS_DIR / "01_topic_analysis" / "tables" / "topic_author_dominance.parquet"

    # Output
    out_dir: Path = RESULTS_DIR

    # Quality filtering controls
    exclude_noise_topics: bool = True
    exclude_author_dominant_topics: bool = True

    # Thresholds for topic gating (topic_health / dominance)
    noise_threshold: float = 0.50          # is_noise_topic < threshold
    author_threshold: float = 0.50         # is_author_topic < threshold

    # Composite-level thresholds
    default_min_topics: int = 4
    core_min_topics: int = 6               # CORE requires >= this many topics (in addition to coverage/stability)
    min_book_coverage: float = 0.20        # at least this fraction of books have non-trivial composite mass
    coverage_min_mass: float = 1e-3        # non-trivial mass threshold for coverage (fixes 'everything > 0')
    warn_if_quality_missing: bool = True   # warn if noise/dominance columns missing or mostly NaN

    # Dimensionality
    pca_pc1_threshold: float = 0.30        # PC1 explained variance

    # Stability diagnostics
    split_half_bootstrap: int = 300        # keep modest for notebook speed
    epsilon: float = 1e-6                 # for log ratios
    make_zscores: bool = True
    make_log_ratios: bool = True

    # Segment configuration
    segment_col: str = "segment"           # expected values: begin/middle/end

    # Saving
    save_intermediate_tables: bool = True

config = PipelineConfig()

print("✅ Measurement pipeline config created.")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("CORRELATION_ANALYSIS_DIR:", CORRELATION_ANALYSIS_DIR)
print("OUT_DIR:", config.out_dir)

# Save run metadata (helps reproducibility)
run_meta = {
    "run_timestamp": datetime.datetime.now().isoformat(),
    "pipeline": "notebook05_measurement_v5",
    "config": {k: str(v) if isinstance(v, Path) else v for k, v in asdict(config).items()},
}
(config.out_dir / "run_metadata.json").write_text(json.dumps(run_meta, indent=2), encoding="utf-8")
print("📝 Saved run_metadata.json")


✅ Measurement pipeline config created.
PROJECT_ROOT: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor
CORRELATION_ANALYSIS_DIR: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/correlation_analysis
OUT_DIR: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/measurement_v5
📝 Saved run_metadata.json


## Section 1 — Load inputs + sanity checks

In [2]:
from pathlib import Path

def read_table(path: Path) -> pd.DataFrame:
    """Read parquet/csv safely; raise with a clear message if missing."""
    if path is None:
        raise ValueError("Path is None.")
    if not Path(path).exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    if str(path).endswith(".parquet"):
        return pd.read_parquet(path)
    elif str(path).endswith(".csv"):
        return pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")

def read_optional_table(path: Optional[Path]) -> Optional[pd.DataFrame]:
    if path is None:
        return None
    if not Path(path).exists():
        return None
    return read_table(path)

# Required
topic_lookup = read_table(config.topic_lookup_path)
book_topic_probs = read_table(config.book_topic_probs_path)

# Optional diagnostics
topic_health = read_optional_table(config.topic_health_path)
author_dom = read_optional_table(config.author_dominance_path)

# Optional segment probs
segment_topic_probs = read_optional_table(config.segment_topic_probs_path)

print("Shapes:")
print("  topic_lookup:", topic_lookup.shape)
print("  book_topic_probs:", book_topic_probs.shape)
print("  topic_health:", None if topic_health is None else topic_health.shape)
print("  author_dom:", None if author_dom is None else author_dom.shape)
print("  segment_topic_probs:", None if segment_topic_probs is None else segment_topic_probs.shape)

# Diagnostic: Inspect quality files
print("\n" + "="*60)
print("QUALITY FILES DIAGNOSTIC")
print("="*60)
if topic_health is not None:
    print(f"\n✓ topic_health loaded: {topic_health.shape}")
    print(f"  Columns: {list(topic_health.columns)}")
    print(f"  Has topic_id: {'topic_id' in topic_health.columns}")
    if 'topic_id' in topic_health.columns:
        print(f"  Topic ID overlap with topic_lookup: {topic_health['topic_id'].isin(topic_lookup['topic_id']).mean():.1%}")
    # Check for noise-related columns
    noise_cols = [c for c in topic_health.columns if 'noise' in c.lower()]
    print(f"  Noise-related columns: {noise_cols}")
    if noise_cols:
        print(f"  Sample values from {noise_cols[0]}:")
        print(topic_health[noise_cols[0]].describe())
else:
    print(f"\n✗ topic_health is None (file may not exist: {config.topic_health_path})")
    print(f"  File exists: {config.topic_health_path.exists() if config.topic_health_path else False}")

if author_dom is not None:
    print(f"\n✓ author_dom loaded: {author_dom.shape}")
    print(f"  Columns: {list(author_dom.columns)}")
    print(f"  Has topic_id: {'topic_id' in author_dom.columns}")
    if 'topic_id' in author_dom.columns:
        print(f"  Topic ID overlap with topic_lookup: {author_dom['topic_id'].isin(topic_lookup['topic_id']).mean():.1%}")
    # Check for author-related columns
    author_cols = [c for c in author_dom.columns if 'author' in c.lower() or 'dominance' in c.lower()]
    print(f"  Author-related columns: {author_cols}")
    if author_cols:
        print(f"  Sample values from {author_cols[0]}:")
        print(author_dom[author_cols[0]].describe())
else:
    print(f"\n✗ author_dom is None (file may not exist: {config.author_dominance_path})")
    print(f"  File exists: {config.author_dominance_path.exists() if config.author_dominance_path else False}")
print("="*60)

# --- Column sanity (minimal, adjust if needed) ---
required_lookup_cols = {"topic_id"}
required_probs_cols = {"book_id", "topic_id"}

missing_lookup = required_lookup_cols - set(topic_lookup.columns)
missing_probs = required_probs_cols - set(book_topic_probs.columns)

if missing_lookup:
    raise ValueError(f"topic_lookup is missing required columns: {missing_lookup}")
if missing_probs:
    raise ValueError(f"book_topic_probs is missing required columns: {missing_probs}")

# Standardize key dtypes
topic_lookup = topic_lookup.copy()
book_topic_probs = book_topic_probs.copy()

topic_lookup["topic_id"] = pd.to_numeric(topic_lookup["topic_id"], errors="raise").astype(int)
book_topic_probs["topic_id"] = pd.to_numeric(book_topic_probs["topic_id"], errors="raise").astype(int)
book_topic_probs["book_id"] = book_topic_probs["book_id"].astype(str)

# Identify probability column name in book_topic_probs
prob_col_candidates = [c for c in book_topic_probs.columns if c.lower() in {"prob", "topic_prob", "topic_probability", "probability"}]
if not prob_col_candidates:
    raise ValueError("Could not find probability column in book_topic_probs (expected one of: prob/probability/topic_prob/...).")
PROB_COL = prob_col_candidates[0]

book_topic_probs[PROB_COL] = pd.to_numeric(book_topic_probs[PROB_COL], errors="coerce").fillna(0.0).clip(lower=0.0)

print("\nQuick integrity checks:")
print("  unique books:", book_topic_probs["book_id"].nunique())
print("  unique topics:", book_topic_probs["topic_id"].nunique())
print("  prob column:", PROB_COL)
print("  topic_id overlap lookup/probs:", book_topic_probs["topic_id"].isin(topic_lookup["topic_id"]).mean())

# Optional: segment sanity
if segment_topic_probs is not None:
    segment_topic_probs = segment_topic_probs.copy()
    # expected columns
    seg_required = {"book_id", "topic_id", config.segment_col}
    seg_missing = seg_required - set(segment_topic_probs.columns)
    if seg_missing:
        raise ValueError(f"segment_topic_probs missing required columns: {seg_missing}")
    seg_prob_candidates = [c for c in segment_topic_probs.columns if c.lower() in {"prob", "topic_prob", "topic_probability", "probability"}]
    if not seg_prob_candidates:
        raise ValueError("Could not find probability column in segment_topic_probs.")
    SEG_PROB_COL = seg_prob_candidates[0]
    segment_topic_probs["topic_id"] = pd.to_numeric(segment_topic_probs["topic_id"], errors="raise").astype(int)
    segment_topic_probs["book_id"] = segment_topic_probs["book_id"].astype(str)
    segment_topic_probs[SEG_PROB_COL] = pd.to_numeric(segment_topic_probs[SEG_PROB_COL], errors="coerce").fillna(0.0).clip(lower=0.0)
    print("  segment prob column:", SEG_PROB_COL)


Shapes:
  topic_lookup: (369, 21)
  book_topic_probs: (33856, 3)
  topic_health: (343, 8)
  author_dom: (324, 15)
  segment_topic_probs: (101568, 5)

QUALITY FILES DIAGNOSTIC

✓ topic_health loaded: (343, 8)
  Columns: ['label', 'prevalence', 'mass', 'concentration_ratio', 'topic_id', 'keywords', 'scene_summary', 'label_is_noise']
  Has topic_id: True
  Topic ID overlap with topic_lookup: 100.0%
  Noise-related columns: ['label_is_noise']
  Sample values from label_is_noise:
count       335
unique        2
top       False
freq        334
Name: label_is_noise, dtype: object

✓ author_dom loaded: (324, 15)
  Columns: ['topic_id', 'label', 'total_books_with_topic', 'n_authors', 'top_author', 'top_author_count', 'top_author_share', 'top2_author_share', 'is_author_driven', 'top_author_tier_concentration', 'author_dominance_flag', 'cliffs_top_trash', 'top_median', 'trash_median', 'prevalence']
  Has topic_id: True
  Topic ID overlap with topic_lookup: 100.0%
  Author-related columns: ['n_aut

## Section 2 — Ontology: composite specifications (theory contract)

In [3]:
from dataclasses import dataclass, field
from typing import Optional, List, Dict
import re

@dataclass(frozen=True)
class CompositeSpec:
    """Theory contract for selecting topics into a composite index."""
    name: str
    taxonomy_main_ids: List[str] = field(default_factory=list)
    include_name_regex: Optional[str] = None
    exclude_name_regex: Optional[str] = None
    include_secondary_cats_any: List[str] = field(default_factory=list)
    # Selection behavior controls (measurement-only)
    semantic_mode: str = "strict"  # {"strict","relaxed","off"}
    fallback_mode: str = "none"   # {"none","within_taxonomy","global_tight"}
    min_topics: int = 4
    allow_global_fallback: bool = False  # legacy; kept for backward compatibility
    note: str = ""

# ------------------------------
# Full COMPOSITES dict (imported from your v4 notebook, unchanged)
# ------------------------------
COMPOSITES: Dict[str, CompositeSpec] = {
    
    # ============================================================
    # SPLIT A: Reassurance/Commitment (was α=-0.22, 80 topics)
    # ============================================================
    "A1_commitment_vows": CompositeSpec(
        name="A1) Commitment & Vows",
        taxonomy_main_ids=["4.5"],  # Reconciliation, Commitments & HEA only
        include_name_regex=r"(commit|promise|forever|vow|marry|marriage|wedding|proposal|ring|engagement|bride|groom|altar)",
        note="explicit commitment language (HEA/marriage plot)",
        min_topics=3,
        allow_global_fallback=False,
    ),
    
    "A2_emotional_safety": CompositeSpec(
        name="A2) Emotional Safety & Reassurance",
        taxonomy_main_ids=["3.1"],  # Positive Emotions & Security
        include_name_regex=r"(safe|reassur|comfort|calm|peace|secur|trust|relax|gentle|sooth|ease|protect|shelter)",
        exclude_name_regex=r"(vow|marry|wed)",  # Prevent overlap with A1
        note="safety/trust/comfort affect",
        min_topics=4,
        allow_global_fallback=False,
    ),
    
    "A3_everyday_tenderness": CompositeSpec(
        name="A3) Everyday Tenderness",
        taxonomy_main_ids=["4.2"],  # Bonding, Everyday Intimacy & Growth
        exclude_name_regex=r"(vow|promise|commit|marry|wed)",  # Exclude A1 formal terms
        include_name_regex=r"(tender|gentle|care|affection|warmth|warm|close|bond|connect|share|cuddle|snuggl|embrace|hold)",
        note="daily affectionate bonding (non-formal)",
        min_topics=6,
        allow_global_fallback=False,
    ),
    
    # ============================================================
    # SPLIT B: Mutual Intimacy (was α=-0.15, 106 topics)
    # ============================================================
    "B1_attraction_chemistry": CompositeSpec(
        name="B1) Attraction & Sexual Chemistry",
        taxonomy_main_ids=["2.1", "2.2"],  # Attraction + Kissing ONLY (no bonding/emotions)
        exclude_name_regex=r"\b(penetrat|clit|pussy|cock|orgasm|anal|fuck)\b",  # Exclude explicit sex
        note="sexual tension + kissing (pre-coital intimacy)",
        min_topics=8,
        allow_global_fallback=False,
    ),
    
    "B2_emotional_intimacy": CompositeSpec(
        name="B2) Emotional Intimacy",
        taxonomy_main_ids=["3.1", "4.2"],  # Positive emotions + bonding (no physical)
        exclude_name_regex=r"\b(penetrat|clit|pussy|cock|orgasm|anal|fuck|kiss|tongue|lip|mouth)\b",
        include_name_regex=r"(share|open|vulner|understand|connect|close|trust|reveal|honest|bare|confid|intimate|deep)",
        note="emotional closeness (non-physical)",
        min_topics=8,
        allow_global_fallback=False,
    ),
    
    # ============================================================
    # KEEP C: Explicit Eroticism (α=-0.53 is EXPECTED)
    # ============================================================
    "C_explicit_eroticism": CompositeSpec(
        name="C) Explicit Eroticism",
        taxonomy_main_ids=["2.3"],  # Explicit Sexual Acts
        note="explicit sexual act language (heterogeneous by design — use in ratios)",
        min_topics=8,
        allow_global_fallback=False,
    ),
    
    # ============================================================
    # KEEP D: Power/Wealth (α=-0.03, acceptable)
    # ============================================================
    "D_power_wealth_luxury": CompositeSpec(
        name="D) Power / Wealth / Luxury",
        taxonomy_main_ids=["5.2", "5.3", "5.1"],  # Friends, Community, Family (power contexts)
        note="billionaire-world saturation",
        min_topics=10,
        allow_global_fallback=False,
    ),
    
    # ============================================================
    # KEEP E: Coercion/Violence (α=0.07, OK)
    # ============================================================
    "E_coercion_brutality_danger": CompositeSpec(
        name="E) Coercion / Brutality / Danger",
        taxonomy_main_ids=["7.2", "3.2"],  # Violence + Negative Emotions
        include_name_regex=r"(threat|violence|weapon|kidnap|coerc|abuse|trauma|tortur|panic|fear|gun|danger|attack|assault|harm|hurt)",
        note="threat + coercion + traumatic texture",
        min_topics=6,
        allow_global_fallback=False,
    ),
    
    # ============================================================
    # SPLIT F: Angst/Negative Affect (was α=-0.25, 55 topics)
    # ============================================================
    "F1_sadness_grief": CompositeSpec(
        name="F1) Sadness & Grief",
        taxonomy_main_ids=["3.2"],  # Negative Emotions only
        include_name_regex=r"(sad|cry|tear|grief|mourn|sorrow|depress|melanchol|weep|sob|miserable|heartbreak)",
        exclude_name_regex=r"(angry|anger|rage|anxious|anxiety|worry)",  # Exclude other affects
        note="sadness affect cluster",
        min_topics=4,
        allow_global_fallback=False,
    ),
    
    "F2_anger_frustration": CompositeSpec(
        name="F2) Anger & Frustration",
        taxonomy_main_ids=["3.2", "3.3"],  # Negative Emotions + Ambivalence
        include_name_regex=r"(angry|anger|rage|furious|fury|frustrat|irritat|resent|bitter|mad|temper|hostile)",
        exclude_name_regex=r"(sad|cry|tear|anxious|anxiety|worry)",  # Exclude other affects
        note="anger/resentment affect",
        min_topics=4,
        allow_global_fallback=False,
    ),
    
    "F3_anxiety_worry": CompositeSpec(
        name="F3) Anxiety & Worry",
        taxonomy_main_ids=["3.2", "3.4"],  # Negative Emotions + Values/Beliefs
        include_name_regex=r"(anxious|anxiety|worry|nervous|stress|tension|uncertain|doubt|dread|apprehens|unease|concern)",
        exclude_name_regex=r"(angry|anger|rage|sad|cry|tear)",  # Exclude other affects
        note="anxiety/worry affect",
        min_topics=4,
        allow_global_fallback=False,
    ),
    
    # ============================================================
    # RELAXED G, L, M: Were Empty (0 topics) — Now Recovered
    # ============================================================
    "G_courtship_rituals_gifts": CompositeSpec(
        name="G) Courtship Rituals / Gifts",
        taxonomy_main_ids=["5.3", "8.1", "8.2"],  # Community + Domestic Spaces + Public Spaces
        include_name_regex=r"(gift|flower|rose|dinner|date|restaurant|candl|wine|party|celebrat|surprise|romantic|anniversary|chocol|bouquet)",
        exclude_name_regex=r"(work|office|business|meeting|board|corporate)",  # Avoid work dinners
        note="ritualized romance behaviors (RELAXED — broader pattern)",
        min_topics=3,  # Lowered threshold from 6
        allow_global_fallback=True,  # ✅ ENABLED — will search ALL topics if needed
    ),
    
    "L_vices_addictions": CompositeSpec(
        name="L) Vices / Addictions",
        taxonomy_main_ids=["6.2", "6.3", "3.2"],  # Work + Distress (addiction/risk context)
        include_name_regex=r"(drink|drank|alcohol|wine|bottle|bar|drunk|hangover|vodka|whiskey|beer|liquor|smoke|smoking|cigarette|cigar|drug|pill|addict|substance|binge|intoxicat)",
        note="substance use & self-destructive risk (RELAXED)",
        min_topics=3,  # Lowered threshold from 4
        allow_global_fallback=True,  # ✅ ENABLED
    ),
    
    "M_health_recovery_growth": CompositeSpec(
        name="M) Health / Recovery / Growth",
        taxonomy_main_ids=["6.5", "4.2"],  # Law/Medicine + Bonding (caretaking overlap)
        include_name_regex=r"(hospital|doctor|nurse|nurs|clinic|therap|medical|medic|injur|injury|pain|hurt|wound|heal|recover|treatment|ambulance|emergency|surgery|patient|sick|ill|health|care)",
        note="healing & protective caretaking (RELAXED)",
        min_topics=3,  # Lowered threshold from 4
        allow_global_fallback=True,  # ✅ ENABLED
    ),
    
    # ============================================================
    # KEEP H, I, J, K: Were Already Functioning (α > 0)
    # ============================================================
    "H_domestic_nesting": CompositeSpec(
        name="H) Domestic Nesting (Home-as-Refuge)",
        taxonomy_main_ids=["4.2", "5.2", "8.1"],  # Bonding + Friends + Domestic Spaces
        note="nest-building and everyday shared life",
        min_topics=10,
        allow_global_fallback=False,
    ),
    
    "I_humor_lightness": CompositeSpec(
        name="I) Humor / Lightness",
        taxonomy_main_ids=["3.1", "4.2", "2.2"],  # Positive Emotions + Bonding + Kissing
        include_name_regex=r"(laugh|joke|teas|banter|sarcasm|funny|smil|humor|wit|playful|giggl|chuckl|amuse|grin)",
        note="comic relief / breezy tone proxies",
        min_topics=6,
        allow_global_fallback=True,  # Humor is safe to search globally (distinct pattern)
    ),
    
    "J_social_support_kin": CompositeSpec(
        name="J) Social Support / Kin",
        taxonomy_main_ids=["4.3", "4.4", "4.5", "5.1"],  # Secrets + Conflict + HEA + Family
        include_name_regex=r"(family|friend|community|parent|child|sister|brother|mother|father|mom|dad|aunt|uncle|cousin|relative|kin)",
        note="stable social buffering around couple",
        min_topics=8,
        allow_global_fallback=False,
    ),
    
    "K_professional_intrusion": CompositeSpec(
        name="K) Professional Intrusion",
        taxonomy_main_ids=["6.1", "6.2", "6.3", "6.5"],  # All work-related categories
        include_name_regex=r"(work|office|deal|contract|meeting|board|law|court|authority|boss|business|corporate|boardroom|CEO|executive|professional|career|job|employ)",
        note="workplace and institutional texture",
        min_topics=6,
        allow_global_fallback=False,
    ),
    
    # ============================================================
    # KEEP N, O, Q: Functioning Composites
    # ============================================================
    "N_separation_reunion": CompositeSpec(
        name="N) Separation / Reunion",
        taxonomy_main_ids=["4.4", "4.5"],  # Conflict + HEA (breakup/makeup arc)
        include_name_regex=r"(break|broke|leave|left|separat|reconcil|reun|reunion|return|distance|apart|forgiv|sorry|apolog|makeup|mend)",
        note="breakup → reunion arc",
        min_topics=8,
        allow_global_fallback=False,
    ),
    
    "O_aesthetics_appearance": CompositeSpec(
        name="O) Aesthetics / Appearance",
        taxonomy_main_ids=["1.1", "5.3"],  # Body Parts + Community (presentation contexts)
        include_name_regex=r"(dress|suit|clothes|clothing|hair|makeup|groom|heels|beautiful|handsome|stubble|lingerie|outfit|fashion|style|appearance|attractive|gorgeous)",
        note="personal appearance & presentation",
        min_topics=4,
        allow_global_fallback=False,
    ),
    
    "Q_miscommunication": CompositeSpec(
        name="Q) Miscommunication & Conflict",
        taxonomy_main_ids=["4.1", "4.3"],  # (4.1 may not exist in taxonomy, but keeping for compatibility) + Secrets
        include_name_regex=r"(secret|misunderstand|miscommunicat|silent|silence|argument|argue|unclear|confus|lie|lied|lying|decei|hidden|conceal)",
        note="secrets, misunderstandings, conflict",
        min_topics=6,
        allow_global_fallback=False,
    ),
    
    "Q_repair": CompositeSpec(
        name="Q) Repair & Reconciliation",
        taxonomy_main_ids=["4.5", "4.2", "3.1"],  # HEA + Bonding + Positive Emotions
        include_secondary_cats_any=[
            "activity:apologizing", 
            "activity:saying_sorry", 
            "activity:love_confession", 
            "activity:vowing",
            "activity:forgiving",
            "activity:reconciling"
        ],
        include_name_regex=r"(apolog|sorry|forgiv|reconcil|confess|admit|explain|resolve|fix|mend|heal)",
        note="apologies, forgiveness, reconciliation",
        min_topics=4,
        allow_global_fallback=False,
    ),
    
    # ============================================================
    # SPLIT R: Protectiveness (α=-0.72, 15 topics — WORST OFFENDER)
    # ============================================================
    "R1_protective_caretaking": CompositeSpec(
        name="R1) Protective Caretaking",
        taxonomy_main_ids=["3.1", "4.2"],  # Positive Emotions + Bonding
        include_name_regex=r"(protect|safe|care|help|comfort|calm|rescu|rescue|soothe|nurt|nurtur|tend|shelter|shield)",
        exclude_name_regex=r"(guard|security|territorial|possess|control|jealous|mine|claim)",  # Exclude alpha terms
        note="tender protective care (nurturing caretaking)",
        min_topics=4,
        allow_global_fallback=False,
    ),
    
    "R2_alpha_guarding": CompositeSpec(
        name="R2) Alpha Guarding",
        taxonomy_main_ids=["3.3", "4.4"],  # Ambivalence + Conflict
        include_name_regex=r"(guard|mine|possess|territorial|control|jealous|watch|shield|defend|claim|own|belong|stake)",
        note="possessive protective aggression (alpha guarding)",
        min_topics=4,
        allow_global_fallback=False,
    ),
    
    # Keep R_jealousy for continuity with H4 hypothesis
    "R_jealousy_possessiveness": CompositeSpec(
        name="R) Jealousy & Possessiveness",
        taxonomy_main_ids=["3.3", "4.4"],  # Ambivalence + Conflict
        include_name_regex=r"(jealous|possess|mine|territorial|claim|obsess|control|stalk|belong|own)",
        note="jealous/possessive conflict (kept for H4 hypothesis continuity)",
        min_topics=4,
        allow_global_fallback=False,
    ),
    
    # ============================================================
    # KEEP S: Scene Anchors (α=0.40 — BEST COMPOSITE)
    # ============================================================
    "S_scene_anchors": CompositeSpec(
        name="S) Scene Anchors (Setting-Rich)",
        taxonomy_main_ids=["5.3", "8.1", "8.2"],  # Community + Domestic + Public Spaces
        note="setting/objects/time/atmosphere topics (for qualitative sampling)",
        min_topics=8,
        allow_global_fallback=False,
    ),
}

print(f'Loaded {len(COMPOSITES)} composite specs.')


Loaded 26 composite specs.


## Section 2.1 — Per-composite overrides (fix sparse constructs without changing the theory contract)

In [4]:
# Overrides let you adjust selection behavior without rewriting the full COMPOSITES dict.
# Use these sparingly and document why.
# Keys must match COMPOSITES keys exactly.

COMPOSITE_OVERRIDES = {
    # These were collapsing to 1–2 topics due to strict label regex.
    # Strategy: keep taxonomy constraint but relax semantics when too few topics survive.
    "A1_commitment_vows": {"semantic_mode": "relaxed", "fallback_mode": "within_taxonomy", "min_topics": 5},
    "O_aesthetics_appearance": {"semantic_mode": "relaxed", "fallback_mode": "within_taxonomy", "min_topics": 5},
    "A2_emotional_safety": {"semantic_mode": "relaxed", "fallback_mode": "within_taxonomy", "min_topics": 5},
    "F2_anger_frustration": {"semantic_mode": "relaxed", "fallback_mode": "within_taxonomy", "min_topics": 5},
    "F3_anxiety_worry": {"semantic_mode": "relaxed", "fallback_mode": "within_taxonomy", "min_topics": 5},
    "R1_protective_caretaking": {"semantic_mode": "relaxed", "fallback_mode": "within_taxonomy", "min_topics": 5},
}

print(f"Loaded {len(COMPOSITE_OVERRIDES)} composite overrides.")

Loaded 6 composite overrides.


## Section 3 — Helper functions (selection + audits + diagnostics)

In [5]:
import numpy as np
import pandas as pd
from typing import Any, Tuple, Optional, Dict, List
from sklearn.decomposition import PCA

# ---------- basic helpers ----------
def _ensure_cols(df: pd.DataFrame, cols: List[str], name: str) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")

def _safe_series_concat(df: pd.DataFrame, cols: List[str]) -> pd.Series:
    s = pd.Series("", index=df.index, dtype="object")
    for c in cols:
        if c in df.columns:
            s = s + " " + df[c].fillna("").astype(str)
    return s.str.strip()

def matches_pattern(text: pd.Series, pattern: Optional[str]) -> pd.Series:
    if pattern is None:
        return pd.Series(True, index=text.index)
    # Suppress pandas warning about regex match groups (patterns may contain parentheses)
    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UserWarning, message=".*match groups.*")
        return text.fillna("").astype(str).str.contains(pattern, case=False, regex=True, na=False)

# ---------- standardize topic meta ----------
def build_topics_meta(topic_lookup: pd.DataFrame,
                      topic_health: Optional[pd.DataFrame],
                      author_dom: Optional[pd.DataFrame]) -> pd.DataFrame:
    """
    Create a standardized topics_meta table used throughout the pipeline.
    Expected in topic_lookup (at minimum): topic_id.
    Optional: taxonomy_main_id, taxonomy_main_name, name, keywords, etc.
    """
    meta = topic_lookup.copy()

    # Standardize common text fields if present
    if "name" not in meta.columns:
        # Try other likely label columns
        for alt in ["topic_label", "label", "topic_name"]:
            if alt in meta.columns:
                meta["name"] = meta[alt]
                break
    if "keywords" not in meta.columns:
        for alt in ["top_words", "repr", "representation", "terms"]:
            if alt in meta.columns:
                meta["keywords"] = meta[alt]
                break

    # Coerce topic_id types BEFORE merging for reliable merges
    meta["topic_id"] = pd.to_numeric(meta["topic_id"], errors="raise").astype(int)
    
    # Merge optional health/dominance if available
    if topic_health is not None:
        if "topic_id" not in topic_health.columns:
            raise ValueError("topic_health must have topic_id")
        # Ensure topic_id types match before merge
        topic_health = topic_health.copy()
        topic_health["topic_id"] = pd.to_numeric(topic_health["topic_id"], errors="raise").astype(int)
        meta = meta.merge(topic_health, on="topic_id", how="left", suffixes=("", "_health"))
    if author_dom is not None:
        if "topic_id" not in author_dom.columns:
            raise ValueError("author_dom must have topic_id")
        # Ensure topic_id types match before merge
        author_dom = author_dom.copy()
        author_dom["topic_id"] = pd.to_numeric(author_dom["topic_id"], errors="raise").astype(int)
        meta = meta.merge(author_dom, on="topic_id", how="left", suffixes=("", "_auth"))

    # --- Normalize common quality column names (if present) ---
    # The pipeline expects: is_noise_topic, is_author_topic in [0,1] where higher = worse.
    rename_map = {}
    
    # Debug: print available columns that might be relevant
    all_cols = list(meta.columns)
    noise_like = [c for c in all_cols if 'noise' in c.lower()]
    author_like = [c for c in all_cols if 'author' in c.lower() or 'dominance' in c.lower()]
    
    # Noise topic columns - check multiple possible names (including actual column names found)
    # Also check for columns with merge suffixes
    noise_candidates = [
        "label_is_noise",  # Actual column name in topic_health
        "label_is_noise_health",  # In case merge added suffix
        "is_noise_topic", "noise_topic", "is_noise", "noise_flag", "topic_is_noise",
        "is_noise_topic_health", "noise", "is_noise_topic_flag", "noise_indicator"
    ]
    for cand in noise_candidates:
        if cand in meta.columns:
            if "is_noise_topic" not in meta.columns:
                rename_map[cand] = "is_noise_topic"
                break  # Use first match
            elif cand != "is_noise_topic":
                # If target already exists, we can drop the duplicate
                break
    
    # Author dominance columns - check multiple possible names (including actual column names found)
    # Prefer is_author_driven or author_dominance_flag, but also check for merge suffixes
    author_candidates = [
        "is_author_driven",  # Actual column name in author_dom (preferred)
        "is_author_driven_auth",  # In case merge added suffix
        "author_dominance_flag",  # Actual column name in author_dom (alternative)
        "author_dominance_flag_auth",  # In case merge added suffix
        "is_author_topic", "author_topic", "is_author_dominant", "author_dominance",
        "topic_is_author", "is_author_topic_auth", "is_author_dom", "author_dom",
        "author_dominance_score", "is_author_dominance", "author_topic_flag"
    ]
    for cand in author_candidates:
        if cand in meta.columns:
            if "is_author_topic" not in meta.columns:
                rename_map[cand] = "is_author_topic"
                break  # Use first match
            elif cand != "is_author_topic":
                # If target already exists, we can drop the duplicate
                break
    
    if rename_map:
        meta = meta.rename(columns=rename_map)
        print(f"✓ Renamed quality columns: {rename_map}")
    elif config.warn_if_quality_missing:
        # Debug output if renaming didn't happen
        if noise_like:
            print(f"  Debug: Found noise-like columns but didn't rename: {noise_like}")
        if author_like:
            print(f"  Debug: Found author-like columns but didn't rename: {author_like}")
    
    # Convert boolean/object columns to numeric [0,1] if needed
    if "is_noise_topic" in meta.columns:
        if meta["is_noise_topic"].dtype == 'object' or meta["is_noise_topic"].dtype == 'bool':
            # Convert True/False or "True"/"False" to 1/0
            meta["is_noise_topic"] = meta["is_noise_topic"].astype(str).str.lower().map({'true': 1, 'false': 0, '1': 1, '0': 0}).fillna(0).astype(float)
        elif meta["is_noise_topic"].dtype in ['int64', 'int32']:
            meta["is_noise_topic"] = meta["is_noise_topic"].astype(float)
    
    if "is_author_topic" in meta.columns:
        if meta["is_author_topic"].dtype == 'object' or meta["is_author_topic"].dtype == 'bool':
            # Convert True/False or "True"/"False" to 1/0
            meta["is_author_topic"] = meta["is_author_topic"].astype(str).str.lower().map({'true': 1, 'false': 0, '1': 1, '0': 0}).fillna(0).astype(float)
        elif meta["is_author_topic"].dtype in ['int64', 'int32']:
            meta["is_author_topic"] = meta["is_author_topic"].astype(float)

    # Warn if quality columns are missing or mostly NaN (means merges likely failed)
    if config.warn_if_quality_missing:
        for col in ["is_noise_topic", "is_author_topic"]:
            if col not in meta.columns:
                print(f"⚠️  Quality column missing: {col}. Quality filter will have no effect.")
            else:
                nan_frac = float(meta[col].isna().mean())
                if nan_frac > 0.90:
                    print(f"⚠️  Quality column {col} is mostly NaN ({nan_frac:.1%}). Merge likely failed; filter may be ineffective.")
                else:
                    print(f"✓ Quality column {col}: non-NaN {(1-nan_frac):.1%}; median={meta[col].median(skipna=True):.3f}")

    return meta

topics_meta = build_topics_meta(topic_lookup, topic_health, author_dom)
print("topics_meta:", topics_meta.shape)

# ---------- selection stages ----------
def filter_by_taxonomy(topics_df: pd.DataFrame,
                       main_ids: List[str],
                       secondary_cats_any: Optional[List[str]] = None) -> pd.DataFrame:
    if not main_ids and not secondary_cats_any:
        return topics_df.copy()
    mask = pd.Series(False, index=topics_df.index)
    if main_ids and "taxonomy_main_id" in topics_df.columns:
        mask |= topics_df["taxonomy_main_id"].astype(str).isin([str(x) for x in main_ids])
    if secondary_cats_any:
        # secondary categories often stored as list-like strings; do contains-any
        if "taxonomy_secondary_names" in topics_df.columns:
            txt = topics_df["taxonomy_secondary_names"].fillna("").astype(str)
            for cat in secondary_cats_any:
                mask |= txt.str.contains(re.escape(str(cat)), case=False, na=False)
    return topics_df[mask]

def filter_by_semantics(topics_df: pd.DataFrame,
                        include_regex: Optional[str],
                        exclude_regex: Optional[str]) -> pd.DataFrame:
    if len(topics_df) == 0:
        return topics_df
    search_text = _safe_series_concat(topics_df, ["name", "keywords"])
    mask = pd.Series(True, index=topics_df.index)
    if include_regex:
        mask &= matches_pattern(search_text, include_regex)
    if exclude_regex:
        mask &= ~matches_pattern(search_text, exclude_regex)
    return topics_df[mask]

def apply_quality_filters(topics_df: pd.DataFrame,
                          exclude_noise: bool,
                          exclude_author_dom: bool,
                          noise_threshold: float,
                          author_threshold: float) -> pd.DataFrame:
    if len(topics_df) == 0:
        return topics_df
    mask = pd.Series(True, index=topics_df.index)
    if exclude_noise and "is_noise_topic" in topics_df.columns:
        mask &= topics_df["is_noise_topic"].fillna(0).astype(float) < noise_threshold
    if exclude_author_dom and "is_author_topic" in topics_df.columns:
        mask &= topics_df["is_author_topic"].fillna(0).astype(float) < author_threshold
    return topics_df[mask]

def select_topics_with_audit(spec: CompositeSpec,
                            topics_df: pd.DataFrame,
                            config: PipelineConfig) -> Tuple[pd.DataFrame, Dict[str, Any], pd.DataFrame]:
    """
    Two-tier selection:
      Tier 1 (STRICT): taxonomy -> (include+exclude semantics) -> quality
      Tier 2 (RELAXED): taxonomy -> (exclude-only semantics OR no semantics) -> quality
      Optional Tier 3 (GLOBAL_TIGHT): global -> (include+exclude semantics) -> quality

    Returns (selected_topics_df, audit_dict, debug_df).
    debug_df logs per-topic stage membership and the tier used.
    """
    stage0 = topics_df

    # Stage 1: taxonomy (always)
    stage1 = filter_by_taxonomy(stage0, spec.taxonomy_main_ids, spec.include_secondary_cats_any)

    # --- Tier 1 (STRICT) ---
    tier_used = "strict"
    stage2 = filter_by_semantics(stage1, spec.include_name_regex, spec.exclude_name_regex)
    stage3 = apply_quality_filters(stage2,
                                   exclude_noise=config.exclude_noise_topics,
                                   exclude_author_dom=config.exclude_author_dominant_topics,
                                   noise_threshold=config.noise_threshold,
                                   author_threshold=config.author_threshold)

    selected = stage3
    fallback_used = False
    relaxed_candidate = None
    global_candidate = None

    # Decide if we need a relaxed tier
    need_more = (len(selected) < spec.min_topics)

    if need_more and spec.semantic_mode in {"relaxed", "off"}:
        # Tier 2 (RELAXED):
        # - keep taxonomy constraint
        # - drop include regex (or drop all semantics if semantic_mode=="off")
        tier_used = "relaxed"
        if spec.semantic_mode == "off":
            relaxed2 = stage1.copy()
        else:
            # exclude-only: don't require literal label match; only remove obvious wrong matches
            relaxed2 = filter_by_semantics(stage1, include_regex=None, exclude_regex=spec.exclude_name_regex)
        relaxed3 = apply_quality_filters(relaxed2,
                                         exclude_noise=config.exclude_noise_topics,
                                         exclude_author_dom=config.exclude_author_dominant_topics,
                                         noise_threshold=config.noise_threshold,
                                         author_threshold=config.author_threshold)
        relaxed_candidate = relaxed3

        if spec.fallback_mode in {"within_taxonomy", "none"}:
            # within_taxonomy fallback == choose relaxed set (still taxonomy-bound)
            selected = relaxed_candidate
            fallback_used = True

    # Tier 3 (GLOBAL_TIGHT) — only if explicitly allowed and still too small
    if len(selected) < spec.min_topics:
        if spec.fallback_mode == "global_tight" or spec.allow_global_fallback:
            tier_used = "global_tight"
            global1 = filter_by_semantics(stage0, spec.include_name_regex, spec.exclude_name_regex)
            global2 = apply_quality_filters(global1,
                                            exclude_noise=config.exclude_noise_topics,
                                            exclude_author_dom=config.exclude_author_dominant_topics,
                                            noise_threshold=config.noise_threshold,
                                            author_threshold=config.author_threshold)
            global_candidate = global2
            selected = global_candidate
            fallback_used = True

    # Ensure deterministic order (topic_id ascending)
    if "topic_id" in selected.columns:
        selected = selected.sort_values("topic_id")

    audit = {
        "composite_key": None,  # filled later
        "composite_name": spec.name,
        "taxonomy_ids": ",".join(spec.taxonomy_main_ids) if spec.taxonomy_main_ids else "",
        "has_regex": bool(spec.include_name_regex),
        "semantic_mode": spec.semantic_mode,
        "fallback_mode": spec.fallback_mode,
        "tier_used": tier_used,
        "global_fallback_allowed": bool(spec.fallback_mode == "global_tight" or spec.allow_global_fallback),
        "global_fallback_used": bool(tier_used == "global_tight"),
        "min_topics": int(spec.min_topics),
        "n_after_taxonomy": int(len(stage1)),
        "n_after_semantics": int(len(stage2)),
        "n_after_quality": int(len(stage3)),
        "final_n": int(len(selected)),
        "note": spec.note,
    }

    # Build debug table over union of stage1 + (relaxed/global candidates)
    cand_frames = [stage1]
    if relaxed_candidate is not None:
        cand_frames.append(relaxed_candidate)
    if global_candidate is not None:
        cand_frames.append(global_candidate)

    cand = pd.concat(cand_frames, axis=0) if cand_frames else stage1
    cand = cand.drop_duplicates(subset=["topic_id"]) if "topic_id" in cand.columns else cand.copy()

    s1 = set(stage1["topic_id"].astype(int).tolist()) if len(stage1) else set()
    s2 = set(stage2["topic_id"].astype(int).tolist()) if len(stage2) else set()
    s3 = set(stage3["topic_id"].astype(int).tolist()) if len(stage3) else set()
    sf = set(selected["topic_id"].astype(int).tolist()) if len(selected) else set()

    rows = []
    if "topic_id" in cand.columns:
        for tid in cand["topic_id"].astype(int).tolist():
            in_tax = tid in s1
            in_sem = tid in s2
            in_qual = tid in s3
            in_sel = tid in sf
            if in_sel:
                dropped = "none"
            elif in_qual:
                dropped = "quality"
            elif in_sem:
                dropped = "quality"
            elif in_tax:
                dropped = "semantics"
            else:
                dropped = "taxonomy"
            rows.append({
                "topic_id": int(tid),
                "in_taxonomy": bool(in_tax),
                "in_semantics": bool(in_sem),
                "in_quality": bool(in_qual),
                "selected_final": bool(in_sel),
                "dropped_stage": dropped,
                "tier_used": tier_used,
                "fallback_used": bool(fallback_used),
            })

    debug_df = pd.DataFrame(rows)
    return selected, audit, debug_df

# ---------- matrices and indices ----------
def pivot_probs(book_topic_probs: pd.DataFrame, prob_col: str) -> pd.DataFrame:
    wide = book_topic_probs.pivot_table(index="book_id",
                                        columns="topic_id",
                                        values=prob_col,
                                        aggfunc="sum").fillna(0.0)
    wide.columns = wide.columns.astype(int)
    return wide

def build_composite_matrix(wide_probs: pd.DataFrame,
                           composite_topics: Dict[str, List[int]],
                           empty_value: float = 0.0) -> pd.DataFrame:
    out = {}
    for comp, tids in composite_topics.items():
        tids_present = [t for t in tids if t in wide_probs.columns]
        if len(tids_present) == 0:
            out[comp] = pd.Series(empty_value, index=wide_probs.index, dtype=float)
        else:
            out[comp] = wide_probs[tids_present].sum(axis=1)
    return pd.DataFrame(out)

def zscore_df(df: pd.DataFrame) -> pd.DataFrame:
    z = df.copy()
    for c in z.columns:
        mu = z[c].mean()
        sd = z[c].std(ddof=0)
        if sd == 0 or np.isnan(sd):
            z[c] = 0.0
        else:
            z[c] = (z[c] - mu) / sd
    return z

# ---------- diagnostics ----------
def cronbach_alpha(X: np.ndarray) -> float:
    """
    Cronbach's alpha for item matrix X (n_samples x n_items).
    Returns np.nan if not defined.
    """
    if X.ndim != 2 or X.shape[1] < 2:
        return np.nan
    item_vars = X.var(axis=0, ddof=1)
    total_var = X.sum(axis=1).var(ddof=1)
    if total_var <= 0:
        return np.nan
    k = X.shape[1]
    return (k / (k - 1)) * (1 - item_vars.sum() / total_var)


def mcdonald_omega_total_onefactor(X: np.ndarray) -> float:
    """
    Approximate McDonald's omega total (ωt) using a single-factor model on standardized items.
    Uses loadings from the first eigenvector of the correlation matrix.

    ωt = ( (Σ λ_j)^2 ) / ( (Σ λ_j)^2 + Σ ψ_j )
    where ψ_j are uniquenesses (1 - communality).

    Notes:
    - This is a pragmatic approximation suitable for stability diagnostics in topic-based composites.
    - Returns np.nan if undefined.
    """
    if X.ndim != 2 or X.shape[1] < 2:
        return np.nan

    # Standardize items
    X = X.astype(float)
    Xs = (X - X.mean(axis=0)) / (X.std(axis=0, ddof=1) + 1e-12)

    # Correlation matrix
    R = np.corrcoef(Xs, rowvar=False)
    if not np.isfinite(R).all():
        return np.nan

    # Eigen-decomposition
    vals, vecs = np.linalg.eigh(R)
    idx = np.argsort(vals)[::-1]
    vals = vals[idx]
    vecs = vecs[:, idx]

    # One-factor loadings
    eig1 = vals[0]
    if eig1 <= 0 or not np.isfinite(eig1):
        return np.nan
    loadings = vecs[:, 0] * np.sqrt(eig1)

    # Fix arbitrary sign: enforce positive sum
    if np.sum(loadings) < 0:
        loadings = -loadings
    loadings = np.abs(loadings)

    communalities = loadings**2
    uniqueness = np.clip(1.0 - communalities, 0.0, 1.0)

    num = (np.sum(loadings))**2
    den = num + np.sum(uniqueness)
    return float(num / den) if den > 0 else np.nan

def pca_scores_df(X: pd.DataFrame, n_components: int = 2) -> Tuple[pd.DataFrame, float, float]:
    """
    Compute PCA scores (PC1/PC2) for a topic-item matrix X (books x topics).
    Returns: (scores_df with columns pc1, pc2), pc1_var, pc2_var
    """
    if X.shape[1] < 2:
        return pd.DataFrame(index=X.index), np.nan, np.nan
    Xz = X.copy().astype(float)
    for c in Xz.columns:
        sd = Xz[c].std(ddof=0)
        Xz[c] = 0.0 if sd == 0 or np.isnan(sd) else (Xz[c] - Xz[c].mean()) / sd

    pca = PCA(n_components=min(n_components, Xz.shape[1]))
    scores = pca.fit_transform(Xz.to_numpy())
    ev = pca.explained_variance_ratio_
    pc1 = float(ev[0]) if len(ev) > 0 else np.nan
    pc2 = float(ev[1]) if len(ev) > 1 else np.nan

    cols = ["pc1", "pc2"][:scores.shape[1]]
    scores_df = pd.DataFrame(scores[:, :len(cols)], index=X.index, columns=cols)
    return scores_df, pc1, pc2

def pca_explained_var(X: np.ndarray, n_components: int = 3) -> Tuple[float, float]:
    if X.ndim != 2 or X.shape[1] < 2:
        return (np.nan, np.nan)
    pca = PCA(n_components=min(n_components, X.shape[1]))
    pca.fit(X)
    ev = pca.explained_variance_ratio_
    pc1 = float(ev[0]) if len(ev) > 0 else np.nan
    pc2 = float(ev[1]) if len(ev) > 1 else np.nan
    return pc1, pc2

def leave_one_out_stability(X: pd.DataFrame) -> float:
    """
    Leave-one-topic-out stability as mean correlation between full composite
    and composite with each topic removed.
    """
    if X.shape[1] < 2:
        return np.nan
    full = X.sum(axis=1)
    cors = []
    for col in X.columns:
        lo = X.drop(columns=[col]).sum(axis=1)
        corr = full.corr(lo)
        if pd.notna(corr):
            cors.append(corr)
    return float(np.mean(cors)) if cors else np.nan

def split_half_stability(X: pd.DataFrame, n_boot: int = 300, seed: int = 7) -> float:
    """
    Split topics into two halves repeatedly; correlate half-sums across books.
    """
    if X.shape[1] < 4:
        return np.nan
    rng = np.random.default_rng(seed)
    cols = list(X.columns)
    cors = []
    for _ in range(n_boot):
        rng.shuffle(cols)
        mid = len(cols)//2
        a = X[cols[:mid]].sum(axis=1)
        b = X[cols[mid:]].sum(axis=1)
        corr = a.corr(b)
        if pd.notna(corr):
            cors.append(corr)
    return float(np.mean(cors)) if cors else np.nan


def build_composite_matrix_max(wide_probs: pd.DataFrame,
                               composite_topics: Dict[str, List[int]],
                               empty_value: float = 0.0) -> pd.DataFrame:
    """
    Alternative aggregation: for each composite, take MAX topic probability across its member topics.
    Useful as a robustness alternative to SUM.
    """
    out = {}
    for comp, tids in composite_topics.items():
        tids_present = [t for t in tids if t in wide_probs.columns]
        if len(tids_present) == 0:
            out[comp] = pd.Series(empty_value, index=wide_probs.index, dtype=float)
        else:
            out[comp] = wide_probs[tids_present].max(axis=1)
    return pd.DataFrame(out)


✓ Renamed quality columns: {'label_is_noise': 'is_noise_topic', 'is_author_driven': 'is_author_topic'}
✓ Quality column is_noise_topic: non-NaN 100.0%; median=0.000
✓ Quality column is_author_topic: non-NaN 100.0%; median=0.000
topics_meta: (369, 43)


## Section 4 — Pipeline runner: run_all(config)

In [6]:
from typing import Any, Dict

def classify_composites(audit_df: pd.DataFrame,
                        coverage_df: pd.DataFrame,
                        diag_df: pd.DataFrame,
                        config: PipelineConfig) -> pd.DataFrame:
    """
    Combine audit + coverage + diagnostics into one composite registry.
    Adds:
      - measure_type: ATOMIC (<3 topics) vs COMPOSITE
      - status_core: CORE vs EXPLORATORY (operationalized)
      - dimensionality: UNIDIMENSIONAL vs MULTIDIMENSIONAL
      - recommended_score: sum / pc1 / pc1+pc2
    """
    reg = audit_df.merge(coverage_df, on="composite_key", how="left").merge(diag_df, on="composite_key", how="left")

    # Measure type
    reg["measure_type"] = np.where(reg["final_n"] < 3, "ATOMIC", "COMPOSITE")

    # CORE gating: require enough topics, coverage, and a meaningful stability/coherence metric.
    # Use omega_total and/or stability_boot_to_full (more informative than leave-one-out for large composites).
    omega_ok = reg["omega_total"].fillna(-np.inf) >= 0.55
    boot_ok = reg["stability_boot_to_full"].fillna(-np.inf) >= 0.60

    reg["status_core"] = np.where(
        (reg["measure_type"] == "COMPOSITE") &
        (reg["final_n"] >= config.core_min_topics) &
        (reg["book_coverage_frac"] >= config.min_book_coverage) &
        (omega_ok | boot_ok),
        "CORE",
        "EXPLORATORY"
    )

    # Dimensionality tag (only meaningful for composites with >=2 topics)
    reg["dimensionality"] = np.where(
        reg["pca_pc1"].fillna(0) >= config.pca_pc1_threshold,
        "UNIDIMENSIONAL",
        "MULTIDIMENSIONAL"
    )

    # Recommended score for downstream modeling
    reg["recommended_score"] = np.where(
        reg["dimensionality"] == "MULTIDIMENSIONAL",
        "pc1",
        "sum"
    )
    # Atomic measures: treat as atomic
    reg.loc[reg["measure_type"] == "ATOMIC", "recommended_score"] = "atomic_sum"

    return reg

def run_all(config: PipelineConfig,
            topics_meta: pd.DataFrame,
            book_topic_probs: pd.DataFrame,
            prob_col: str,
            segment_topic_probs: Optional[pd.DataFrame] = None,
            segment_prob_col: Optional[str] = None) -> Dict[str, Any]:
    """
    Orchestrates the entire measurement pipeline and returns a dict of outputs.
    """
    # 1) Select membership with stage-wise audits
    composite_topics: Dict[str, List[int]] = {}
    audit_rows: List[Dict[str, Any]] = []
    debug_rows: List[pd.DataFrame] = []

    for key, spec in COMPOSITES.items():
        ov = COMPOSITE_OVERRIDES.get(key, {}) if 'COMPOSITE_OVERRIDES' in globals() else {}
        spec2 = CompositeSpec(
            name=spec.name,
            taxonomy_main_ids=spec.taxonomy_main_ids,
            include_name_regex=spec.include_name_regex,
            exclude_name_regex=spec.exclude_name_regex,
            include_secondary_cats_any=spec.include_secondary_cats_any,
            semantic_mode=str(ov.get('semantic_mode', getattr(spec, 'semantic_mode', 'strict'))),
            fallback_mode=str(ov.get('fallback_mode', getattr(spec, 'fallback_mode', 'none'))),
            min_topics=int(ov.get('min_topics', int(spec.min_topics) if spec.min_topics else config.default_min_topics)),
            allow_global_fallback=bool(spec.allow_global_fallback),
            note=spec.note,
        )
        selected_df, audit, debug_df = select_topics_with_audit(spec2, topics_meta, config)
        tids = selected_df["topic_id"].astype(int).tolist() if len(selected_df) else []
        composite_topics[key] = tids
        audit["composite_key"] = key
        audit_rows.append(audit)
        debug_df = debug_df.copy()
        debug_df["composite_key"] = key
        debug_rows.append(debug_df)

    audit_df = pd.DataFrame(audit_rows).sort_values(["final_n", "composite_key"], ascending=[True, True]).reset_index(drop=True)

    # 2) Membership table (long)
    membership_rows = []
    for comp, tids in composite_topics.items():
        for tid in tids:
            row = {"composite_key": comp, "topic_id": int(tid)}
            # attach labels if present
            trow = topics_meta.loc[topics_meta["topic_id"] == int(tid)]
            if len(trow):
                for col in ["name", "keywords", "taxonomy_main_id", "taxonomy_main_name"]:
                    if col in trow.columns:
                        row[col] = trow.iloc[0][col]
            membership_rows.append(row)
    membership_df = pd.DataFrame(membership_rows)

    # Debug table (per-topic stage pass/fail)
    membership_debug_df = pd.concat(debug_rows, ignore_index=True) if debug_rows else pd.DataFrame()

    # 3) Book-level matrices
    wide = pivot_probs(book_topic_probs, prob_col)
    comp_sum_raw = build_composite_matrix(wide, composite_topics, empty_value=0.0)
    comp_sum_z = zscore_df(comp_sum_raw) if config.make_zscores else comp_sum_raw.copy()

    comp_max_raw = build_composite_matrix_max(wide, composite_topics, empty_value=0.0)
    comp_max_z = zscore_df(comp_max_raw) if config.make_zscores else comp_max_raw.copy()

    # 4) Coverage (fraction of books with composite mass > 0)
    coverage_rows = []
    for comp in comp_sum_raw.columns:
        frac = float((comp_sum_raw[comp] > config.coverage_min_mass).mean())
        coverage_rows.append({"composite_key": comp, "book_coverage_frac": frac})
    coverage_df = pd.DataFrame(coverage_rows)

    # 5) Diagnostics per composite (alpha, PCA, stability)
    diag_rows = []
    pca_score_blocks = []  # long form: book_id, composite_key, pc1, pc2
    for comp, tids in composite_topics.items():
        tids_present = [t for t in tids if t in wide.columns]
        if len(tids_present) < 2:
            diag_rows.append({
                "composite_key": comp,
                "alpha": np.nan,
                "pca_pc1": np.nan,
                "pca_pc2": np.nan,
                "stability_leave_one_out": np.nan,
                "stability_split_half": np.nan,
                "omega_total": np.nan,
                "mean_intertopic_corr": np.nan,
                "stability_boot_to_full": np.nan,
            })
            continue
        X = wide[tids_present]
        X_np = X.to_numpy()
        alpha = cronbach_alpha(X_np)
        omega = mcdonald_omega_total_onefactor(X_np)
        pc1, pc2 = pca_explained_var(X_np)
        # also store PCA scores (PC1/PC2) per book for multidimensional handling
        scores_df, _, _ = pca_scores_df(X)
        if len(scores_df):
            tmp = scores_df.reset_index().rename(columns={"index":"book_id"})
            tmp.insert(1, "composite_key", comp)
            pca_score_blocks.append(tmp)

            # Mean inter-topic correlation (internal coherence signal; can be negative under compositional constraints)
        R = np.corrcoef(X.to_numpy(), rowvar=False)
        mean_corr = np.nan
        if R.shape[0] >= 2 and np.isfinite(R).any():
            iu = np.triu_indices(R.shape[0], k=1)
            vals = R[iu]
            vals = vals[np.isfinite(vals)]
            mean_corr = float(np.mean(vals)) if len(vals) else np.nan

        # Bootstrap subsample stability: correlate full composite with subsample-of-topics composite
        def _bootstrap_to_full(Xdf: pd.DataFrame, n_boot: int = 200, frac: float = 0.5, seed: int = 11) -> float:
            if Xdf.shape[1] < 4:
                return np.nan
            rng = np.random.default_rng(seed)
            cols = list(Xdf.columns)
            full = Xdf.sum(axis=1)
            cors = []
            k = max(2, int(np.ceil(len(cols) * frac)))
            for _ in range(n_boot):
                sub = rng.choice(cols, size=k, replace=False)
                sub_sum = Xdf[list(sub)].sum(axis=1)
                c = full.corr(sub_sum)
                if pd.notna(c):
                    cors.append(c)
            return float(np.mean(cors)) if cors else np.nan

        boot_to_full = _bootstrap_to_full(X, n_boot=min(200, config.split_half_bootstrap))

        loo = leave_one_out_stability(X)
        sh = split_half_stability(X, n_boot=config.split_half_bootstrap)
        diag_rows.append({
            "composite_key": comp,
            "alpha": float(alpha) if alpha==alpha else np.nan,
            "omega_total": float(omega) if omega==omega else np.nan,
            "pca_pc1": float(pc1) if pc1==pc1 else np.nan,
            "pca_pc2": float(pc2) if pc2==pc2 else np.nan,
            "stability_leave_one_out": float(loo) if loo==loo else np.nan,
            "stability_split_half": float(sh) if sh==sh else np.nan,
            "mean_intertopic_corr": mean_corr,
            "stability_boot_to_full": float(boot_to_full) if boot_to_full==boot_to_full else np.nan,
        })
    diag_df = pd.DataFrame(diag_rows)
    pca_scores_long_df = pd.concat(pca_score_blocks, ignore_index=True) if pca_score_blocks else pd.DataFrame()

    # 6) Composite registry (CORE/EXPLORATORY, dimensionality)
    registry_df = classify_composites(audit_df, coverage_df, diag_df, config).sort_values(["status_core","final_n","composite_key"], ascending=[True, True, True])

    # 7) Derived indices (outcome-blind): keep your existing v4 logic, but make it pure
    def add_derived_indices(df: pd.DataFrame, epsilon: float, make_log_ratios: bool) -> pd.DataFrame:
        """
        Add derived indices to a DataFrame of composite indices.

        NOTE: All derived-index formulas are copied from Notebook 5 (v4) Cell 8.
        This is measurement-only and outcome-blind.
        """
        out = df.copy()

        # Helper to get column or NaN series
        def col(name: str) -> pd.Series:
            return out[name] if name in out.columns else pd.Series(np.nan, index=out.index)

        def log_ratio(
            numerator: pd.Series,
            denominator: pd.Series,
            epsilon: float = epsilon
        ) -> pd.Series:
            """
            Compute log(numerator / denominator) safely.

            Formula: log(num/denom) = log(num + ε) - log(denom + ε)

            Args:
                numerator: Series of numerator values
                denominator: Series of denominator values
                epsilon: Small constant to prevent log(0)

            Returns:
                Series of log-ratio values (NaN if either input is NaN)
            """
            # Add epsilon to prevent log(0)
            num_safe = numerator.fillna(0) + epsilon
            denom_safe = denominator.fillna(0) + epsilon

            # Suppress pandas/numpy RuntimeWarnings for invalid log/zero divides
            with np.errstate(invalid='ignore', divide='ignore'):
                result = np.log(num_safe) - np.log(denom_safe)

            # Preserve NaN where either input was NaN
            result[numerator.isna() | denominator.isna()] = np.nan

            return result


        # ============================================================
        # STEP 1: Reconstruct totals from splits (backward compatibility)
        # ============================================================

        # A_total = sum of A splits
        A1 = col("A1_commitment_vows")
        A2 = col("A2_emotional_safety")
        A3 = col("A3_everyday_tenderness")
        A_total = A1.fillna(0) + A2.fillna(0) + A3.fillna(0)
        A_total[A_total == 0] = np.nan  # Preserve NaN if all components missing

        # B_total = sum of B splits
        B1 = col("B1_attraction_chemistry")
        B2 = col("B2_emotional_intimacy")
        B_total = B1.fillna(0) + B2.fillna(0)
        B_total[B_total == 0] = np.nan

        # F_total = sum of F splits
        F1 = col("F1_sadness_grief")
        F2 = col("F2_anger_frustration")
        F3 = col("F3_anxiety_worry")
        F_total = F1.fillna(0) + F2.fillna(0) + F3.fillna(0)
        F_total[F_total == 0] = np.nan

        # Rp_total = sum of R splits (protective caretaking + alpha guarding)
        R1 = col("R1_protective_caretaking")
        R2 = col("R2_alpha_guarding")
        Rp_total = R1.fillna(0) + R2.fillna(0)
        Rp_total[Rp_total == 0] = np.nan

        # Other composites (not split)
        C = col("C_explicit_eroticism")
        D = col("D_power_wealth_luxury")
        E = col("E_coercion_brutality_danger")
        Qm = col("Q_miscommunication")
        Qr = col("Q_repair")
        Rj = col("R_jealousy_possessiveness")


        # ============================================================
        # STEP 2: Create original derived indices (using totals)
        # ============================================================

        if make_log_ratios:
            # H1: Love/sex balance
            out["H1_love_over_sex_log"] = log_ratio((A_total + B_total), C)

        # H3: Luxury × intimacy interaction
        out["H3_luxury_saturation"] = D
        out["H3_love_depth"] = (A_total + B_total)
        out["H3_interaction"] = out["H3_luxury_saturation"] * out["H3_love_depth"]

        if make_log_ratios:
            # H4: Protectiveness balance
            out["H4_protect_over_jealous_log"] = log_ratio(Rp_total, Rj)

        if make_log_ratios:
            # H5: Dark/tender balance
            out["H5_dark_over_tender_log"] = log_ratio((E + F_total), B_total)

        if make_log_ratios:
            # Q: Repair/miscommunication
            out["Q_repair_over_miscomm_log"] = log_ratio(Qr, Qm)

        # ============================================================
        # STEP 3: Create NEW variant indices (using specific splits)
        # ============================================================

        if make_log_ratios:
            # H1 variants: Test different types of intimacy vs. sex
            out["H1_chemistry_over_sex_log"] = log_ratio(B1, C)

            out["H1_tender_over_sex_log"] = log_ratio((A2 + A3), C)

            out["H1_vows_over_sex_log"] = log_ratio(A1, C)

        if make_log_ratios:
            # H4 variant: Tender care vs. possessive guarding
            out["H4_tender_over_alpha_log"] = log_ratio(R1, R2)

        if make_log_ratios:
            # H5 variants: Different types of darkness
            out["H5_trauma_over_safety_log"] = log_ratio((E + F1), A2)

            out["H5_anger_over_tender_log"] = log_ratio(F2, B_total)

        return out

    book_indices_sum_raw = comp_sum_raw.copy()
    book_indices_sum_z = comp_sum_z.copy()

    book_indices_max_raw = comp_max_raw.copy()
    book_indices_max_z = comp_max_z.copy()

    book_indices_sum_raw = add_derived_indices(book_indices_sum_raw, config.epsilon, config.make_log_ratios)
    # Standardize AFTER derived indices are added (so derived predictors are z-scored too)
    book_indices_sum_z = zscore_df(book_indices_sum_raw) if config.make_zscores else book_indices_sum_raw.copy()

    book_indices_max_raw = add_derived_indices(book_indices_max_raw, config.epsilon, config.make_log_ratios)
    book_indices_max_z = zscore_df(book_indices_max_raw) if config.make_zscores else book_indices_max_raw.copy()

    # 8) Segment-level indices (begin/middle/end), no END selection
    segment_indices_sum_raw = None
    segment_indices_sum_z = None
    segment_indices_max_raw = None
    segment_indices_max_z = None
    arc_contrasts_sum = None
    arc_contrasts_max = None
    if segment_topic_probs is not None:
        if segment_prob_col is None:
            raise ValueError("segment_prob_col is required when segment_topic_probs is provided.")
        seg_wide = segment_topic_probs.pivot_table(index=["book_id", config.segment_col],
                                                   columns="topic_id",
                                                   values=segment_prob_col,
                                                   aggfunc="sum").fillna(0.0)
        seg_wide.columns = seg_wide.columns.astype(int)

        # Compute composite SUM and MAX per (book, segment)
        seg_sum_out = {}
        seg_max_out = {}
        for comp, tids in composite_topics.items():
            tids_present = [t for t in tids if t in seg_wide.columns]
            if len(tids_present) == 0:
                seg_sum_out[comp] = pd.Series(0.0, index=seg_wide.index, dtype=float)
                seg_max_out[comp] = pd.Series(0.0, index=seg_wide.index, dtype=float)
            else:
                seg_sum_out[comp] = seg_wide[tids_present].sum(axis=1)
                seg_max_out[comp] = seg_wide[tids_present].max(axis=1)
        segment_indices_sum_raw = pd.DataFrame(seg_sum_out, index=seg_wide.index).reset_index()
        segment_indices_max_raw = pd.DataFrame(seg_max_out, index=seg_wide.index).reset_index()
        # Zscore within entire sample (not within each segment)
        seg_vals = segment_indices_sum_raw.drop(columns=["book_id", config.segment_col])
        seg_z = zscore_df(seg_vals) if config.make_zscores else seg_vals
        segment_indices_sum_z = pd.concat([segment_indices_sum_raw[["book_id", config.segment_col]], seg_z], axis=1)

        # Z for MAX variant
        seg_vals2 = segment_indices_max_raw.drop(columns=["book_id", config.segment_col])
        seg_z2 = zscore_df(seg_vals2) if config.make_zscores else seg_vals2
        segment_indices_max_z = pd.concat([segment_indices_max_raw[["book_id", config.segment_col]], seg_z2], axis=1)

        # Arc contrasts (outcome-blind): end-begin, middle-begin, per composite
        def _arc_from_seg(seg_df: pd.DataFrame) -> pd.DataFrame:
            # seg_df columns: book_id, segment, comps...
            piv = seg_df.pivot_table(index="book_id", columns=config.segment_col, values=[c for c in seg_df.columns if c not in {"book_id", config.segment_col}], aggfunc="sum")
            # Flatten MultiIndex columns: (comp, segment)
            piv.columns = [f"{comp}__{seg}" for comp, seg in piv.columns]
            piv = piv.reset_index()
            out = {"book_id": piv["book_id"]}
            for comp in [c for c in seg_df.columns if c not in {"book_id", config.segment_col}]:
                b = piv.get(f"{comp}__begin")
                m = piv.get(f"{comp}__middle")
                e = piv.get(f"{comp}__end")
                if b is not None and e is not None:
                    out[f"{comp}__end_minus_begin"] = e - b
                if b is not None and m is not None:
                    out[f"{comp}__mid_minus_begin"] = m - b
            return pd.DataFrame(out)

        arc_contrasts_sum = _arc_from_seg(segment_indices_sum_raw)
        arc_contrasts_max = _arc_from_seg(segment_indices_max_raw)

        # Arc contrasts (outcome-blind): end-begin, middle-begin
        def _arc(df_long: pd.DataFrame) -> pd.DataFrame:
            idx_cols = ["book_id", config.segment_col]
            val_cols = [c for c in df_long.columns if c not in idx_cols]
            wide_seg = df_long.pivot_table(index="book_id", columns=config.segment_col, values=val_cols, aggfunc="first")
            # columns become multiindex (val, segment)
            arcs = {}
            for v in val_cols:
                if (v, "end") in wide_seg.columns and (v, "begin") in wide_seg.columns:
                    arcs[f"{v}__end_minus_begin"] = wide_seg[(v, "end")] - wide_seg[(v, "begin")]
                if (v, "middle") in wide_seg.columns and (v, "begin") in wide_seg.columns:
                    arcs[f"{v}__middle_minus_begin"] = wide_seg[(v, "middle")] - wide_seg[(v, "begin")]
            return pd.DataFrame(arcs).reset_index()

        arc_contrasts_sum = _arc(segment_indices_sum_raw)
        arc_contrasts_max = _arc(segment_indices_max_raw)


    outputs = {
        "topics_meta": topics_meta,
        "composite_topics": composite_topics,
        "membership_df": membership_df,
        "membership_debug_df": membership_debug_df,
        "audit_df": audit_df,
        "coverage_df": coverage_df,
        "diag_df": diag_df,
        "pca_scores_long_df": pca_scores_long_df,
        "registry_df": registry_df,
        "book_indices_raw": book_indices_sum_raw.reset_index().rename(columns={"index":"book_id"}) if book_indices_sum_raw.index.name=="book_id" else book_indices_sum_raw.reset_index().rename(columns={"index":"book_id"}),
        "book_indices_z": book_indices_sum_z.reset_index().rename(columns={"index":"book_id"}) if book_indices_sum_z.index.name=="book_id" else book_indices_sum_z.reset_index().rename(columns={"index":"book_id"}),
        "book_indices_max_raw": book_indices_max_raw.reset_index().rename(columns={"index":"book_id"}) if book_indices_max_raw.index.name=="book_id" else book_indices_max_raw.reset_index().rename(columns={"index":"book_id"}),
        "book_indices_max_z": book_indices_max_z.reset_index().rename(columns={"index":"book_id"}) if book_indices_max_z.index.name=="book_id" else book_indices_max_z.reset_index().rename(columns={"index":"book_id"}),
        "book_wide": wide.reset_index(),
        "segment_indices_raw": segment_indices_sum_raw,
        "segment_indices_z": segment_indices_sum_z,
        "segment_indices_max_raw": segment_indices_max_raw,
        "segment_indices_max_z": segment_indices_max_z,
        "arc_contrasts_sum": arc_contrasts_sum,
        "arc_contrasts_max": arc_contrasts_max,
    }
    return outputs

# Run pipeline
outputs = run_all(
    config=config,
    topics_meta=topics_meta,
    book_topic_probs=book_topic_probs,
    prob_col=PROB_COL,
    segment_topic_probs=segment_topic_probs,
    segment_prob_col=SEG_PROB_COL if segment_topic_probs is not None else None
)

print("✅ Pipeline run complete.")
print("Outputs keys:", sorted(list(outputs.keys())))


✅ Pipeline run complete.
Outputs keys: ['arc_contrasts_max', 'arc_contrasts_sum', 'audit_df', 'book_indices_max_raw', 'book_indices_max_z', 'book_indices_raw', 'book_indices_z', 'book_wide', 'composite_topics', 'coverage_df', 'diag_df', 'membership_debug_df', 'membership_df', 'pca_scores_long_df', 'registry_df', 'segment_indices_max_raw', 'segment_indices_max_z', 'segment_indices_raw', 'segment_indices_z', 'topics_meta']


## Section 5 — Quick review: audits, registry, and example outputs

In [7]:
# Pipeline audit (counts per stage)
display(outputs["audit_df"].head(15))

print("\nComposite registry (CORE/EXPLORATORY, dimensionality, stability):")
display(outputs["registry_df"].head(20))

print("\nMembership table (first rows):")
display(outputs["membership_df"].head(20))

print("\nMembership DEBUG (first rows):")
display(outputs["membership_debug_df"].head(20))

print("\nPCA scores (long, first rows):")
display(outputs["pca_scores_long_df"].head(10))

print("\nBook indices (raw, preview):")
display(outputs["book_indices_raw"].head())

print("\nBook indices (MAX aggregation, preview):")
display(outputs["book_indices_max_raw"].head())

if outputs["segment_indices_raw"] is not None:
    print("\nSegment indices (raw, preview):")
    display(outputs["segment_indices_raw"].head())


if outputs.get("arc_contrasts_sum") is not None:
    print("\nArc contrasts (SUM, preview):")
    display(outputs["arc_contrasts_sum"].head())


,composite_key,composite_name,taxonomy_ids,has_regex,semantic_mode,fallback_mode,tier_used,global_fallback_allowed,global_fallback_used,min_topics,n_after_taxonomy,n_after_semantics,n_after_quality,final_n,note
0,A1_commitment_vows,A1) Commitment & Vows,4.5,True,relaxed,within_taxonomy,relaxed,False,False,5,3,1,1,3,explicit commitment language (HEA/marriage plot)
1,G_courtship_rituals_gifts,G) Courtship Rituals / Gifts,"5.3,8.1,8.2",True,strict,none,strict,True,False,3,26,3,3,3,ritualized romance behaviors (RELAXED — broade...
2,A3_everyday_tenderness,A3) Everyday Tenderness,4.2,True,strict,none,strict,False,False,6,71,5,4,4,daily affectionate bonding (non-formal)
3,F1_sadness_grief,F1) Sadness & Grief,3.2,True,strict,none,strict,False,False,4,36,4,4,4,sadness affect cluster
4,R_jealousy_possessiveness,R) Jealousy & Possessiveness,"3.3,4.4",True,strict,none,strict,False,False,4,64,6,4,4,jealous/possessive conflict (kept for H4 hypot...
5,B2_emotional_intimacy,B2) Emotional Intimacy,"3.1,4.2",True,strict,none,strict,False,False,8,83,5,5,5,emotional closeness (non-physical)
6,E_coercion_brutality_danger,E) Coercion / Brutality / Danger,"7.2,3.2",True,strict,none,strict,False,False,6,44,5,5,5,threat + coercion + traumatic texture
7,N_separation_reunion,N) Separation / Reunion,"4.4,4.5",True,strict,none,strict,False,False,8,64,6,5,5,breakup → reunion arc
8,O_aesthetics_appearance,O) Aesthetics / Appearance,"1.1,5.3",True,relaxed,within_taxonomy,relaxed,False,False,5,6,1,1,5,personal appearance & presentation
9,R2_alpha_guarding,R2) Alpha Guarding,"3.3,4.4",True,strict,none,strict,False,False,4,64,9,6,6,possessive protective aggression (alpha guarding)



Composite registry (CORE/EXPLORATORY, dimensionality, stability):


,composite_key,composite_name,taxonomy_ids,has_regex,semantic_mode,fallback_mode,tier_used,global_fallback_allowed,global_fallback_used,min_topics,...,pca_pc1,pca_pc2,stability_leave_one_out,stability_split_half,mean_intertopic_corr,stability_boot_to_full,measure_type,status_core,dimensionality,recommended_score
9,R2_alpha_guarding,R2) Alpha Guarding,"3.3,4.4",True,strict,none,strict,False,False,4,...,0.534264,0.169130,0.932862,0.177146,0.078308,0.754067,COMPOSITE,CORE,UNIDIMENSIONAL,sum
10,Q_repair,Q) Repair & Reconciliation,"4.5,4.2,3.1",True,strict,none,strict,False,False,4,...,0.381219,0.263564,0.939031,0.120486,0.038171,0.780724,COMPOSITE,CORE,UNIDIMENSIONAL,sum
11,M_health_recovery_growth,M) Health / Recovery / Growth,"6.5,4.2",True,strict,none,strict,True,False,3,...,0.385125,0.278011,0.905235,-0.204486,-0.040564,0.609127,COMPOSITE,CORE,UNIDIMENSIONAL,sum
12,I_humor_lightness,I) Humor / Lightness,"3.1,4.2,2.2",True,strict,none,strict,True,False,6,...,0.412485,0.159254,0.942885,0.023717,0.006341,0.732796,COMPOSITE,CORE,UNIDIMENSIONAL,sum
13,Q_miscommunication,Q) Miscommunication & Conflict,"4.1,4.3",True,strict,none,strict,False,False,6,...,0.612443,0.165957,0.928747,-0.064511,0.012731,0.668858,COMPOSITE,CORE,UNIDIMENSIONAL,sum
14,A2_emotional_safety,A2) Emotional Safety & Reassurance,3.1,True,relaxed,within_taxonomy,relaxed,False,False,5,...,0.250555,0.212495,0.978474,0.350504,0.107490,0.816036,COMPOSITE,CORE,MULTIDIMENSIONAL,pc1
15,L_vices_addictions,L) Vices / Addictions,"6.2,6.3,3.2",True,strict,none,global_tight,True,True,3,...,0.462655,0.274515,0.953560,-0.090218,-0.005141,0.654575,COMPOSITE,CORE,UNIDIMENSIONAL,sum
16,K_professional_intrusion,K) Professional Intrusion,"6.1,6.2,6.3,6.5",True,strict,none,strict,False,False,6,...,0.375393,0.134031,0.978513,0.213815,0.056914,0.775467,COMPOSITE,CORE,UNIDIMENSIONAL,sum
17,C_explicit_eroticism,C) Explicit Eroticism,2.3,False,strict,none,strict,False,False,8,...,0.800077,0.125563,0.953633,-0.166113,0.027063,0.620855,COMPOSITE,CORE,UNIDIMENSIONAL,sum
18,S_scene_anchors,S) Scene Anchors (Setting-Rich),"5.3,8.1,8.2",False,strict,none,strict,False,False,8,...,0.450443,0.246238,0.987807,0.314006,0.025028,0.800490,COMPOSITE,CORE,UNIDIMENSIONAL,sum



Membership table (first rows):


,composite_key,topic_id,name,keywords,taxonomy_main_id,taxonomy_main_name
0,A1_commitment_vows,9,Marriage Ceremony Planning,"married, wedding, marriage, marry, divorce, ma...",4.5,"Reconciliation, Commitments & HEA"
1,A1_commitment_vows,61,Apology And Forgiveness,"apologize, forgive, sorry, apology, apologized...",4.5,"Reconciliation, Commitments & HEA"
2,A1_commitment_vows,163,Redemption Pursuit,"yoursa, youa, whata, redemption, contemporary,...",4.5,"Reconciliation, Commitments & HEA"
3,A2_emotional_safety,29,Time Perception Distortion,"minutes, weeks, months, days, week, hours, yea...",3.1,Positive Emotions & Security
4,A2_emotional_safety,48,Deep Breaths During Emotional Moment,"breath, lungs, deep, exhale, inhale, breathing...",3.1,Positive Emotions & Security
5,A2_emotional_safety,85,Pride In Accomplishments,"did, tried, thought, try, trya, course, dida, ...",3.1,Positive Emotions & Security
6,A2_emotional_safety,178,Pursuit Of Happiness,"happiness, happier, make, makes, satisfaction,...",3.1,Positive Emotions & Security
7,A2_emotional_safety,182,Cheek Heating Conversation,"tough, umma, jerky, battle, dig, cheeks, theya...",3.1,Positive Emotions & Security
8,A2_emotional_safety,190,Religious Piety Display,"et, religion, priest, deity, salvation, religi...",3.1,Positive Emotions & Security
9,A2_emotional_safety,192,Optimistic Reassurance,"everythinga, ita, perfectly, suited, going, sa...",3.1,Positive Emotions & Security


...

,book_id,A1_commitment_vows,A2_emotional_safety,A3_everyday_tenderness,B1_attraction_chemistry,B2_emotional_intimacy,C_explicit_eroticism,D_power_wealth_luxury,E_coercion_brutality_danger,F1_sadness_grief,...,H3_interaction,H4_protect_over_jealous_log,H5_dark_over_tender_log,Q_repair_over_miscomm_log,H1_chemistry_over_sex_log,H1_tender_over_sex_log,H1_vows_over_sex_log,H4_tender_over_alpha_log,H5_trauma_over_safety_log,H5_anger_over_tender_log
0,104659050,0.011754,0.006018,0.005406,0.008098,0.002446,0.021019,0.003757,0.003948,0.003948,...,0.000127,1.206868,0.801686,-0.245306,-0.953749,-0.609722,-0.581202,0.851562,0.271504,-0.300655
1,11266880,0.003596,0.006323,0.004029,0.006214,0.002527,0.011528,0.004031,0.002585,0.002585,...,0.000091,0.911726,1.405577,-0.013358,-0.617933,-0.107586,-1.164835,0.398004,-0.201225,0.555754
2,123257687,0.012243,0.007879,0.003311,0.003917,0.003311,0.003988,0.011804,0.002123,0.001862,...,0.000362,0.835788,1.511558,0.467357,-0.018054,1.031523,1.121462,0.267579,-0.681291,0.688782
3,123446478,0.003999,0.007473,0.003553,0.004612,0.002352,0.010844,0.003797,0.002669,0.002669,...,0.000083,1.440174,1.231659,0.456270,-0.854937,0.016618,-0.997480,1.169897,-0.336431,0.285419
4,127305713,0.001919,0.007619,0.002852,0.004631,0.002852,0.005037,0.009058,0.002147,0.002147,...,0.000180,1.671259,0.983997,0.608101,-0.083875,0.731734,-0.964518,1.463048,-0.573334,0.049514



Segment indices (raw, preview):


,book_id,segment,A1_commitment_vows,A2_emotional_safety,A3_everyday_tenderness,B1_attraction_chemistry,B2_emotional_intimacy,C_explicit_eroticism,D_power_wealth_luxury,E_coercion_brutality_danger,...,J_social_support_kin,K_professional_intrusion,N_separation_reunion,O_aesthetics_appearance,Q_miscommunication,Q_repair,R1_protective_caretaking,R2_alpha_guarding,R_jealousy_possessiveness,S_scene_anchors
0,104659050,begin,0.002842,0.045896,0.006991,0.118413,0.002207,0.098375,0.058783,0.011556,...,0.031781,0.036146,0.002417,0.002829,0.028892,0.035774,0.164351,0.002621,0.001367,0.106633
1,104659050,end,0.003627,0.015649,0.005624,0.098336,0.001666,0.102622,0.072278,0.041645,...,0.078964,0.047257,0.005723,0.004017,0.040858,0.068983,0.151001,0.001849,0.000965,0.097303
2,104659050,middle,0.006252,0.036955,0.002634,0.097086,0.001440,0.134350,0.059648,0.004610,...,0.047293,0.110640,0.001483,0.049579,0.017161,0.074291,0.199355,0.036814,0.036111,0.079180
3,11266880,begin,0.000507,0.078507,0.001000,0.094357,0.008818,0.127574,0.066954,0.022712,...,0.030979,0.045828,0.012579,0.036085,0.012924,0.062718,0.162419,0.019728,0.018422,0.098603
4,11266880,end,0.001992,0.076547,0.018560,0.121120,0.015664,0.088163,0.093156,0.007661,...,0.036340,0.050184,0.003147,0.012498,0.012655,0.048552,0.181712,0.003007,0.001627,0.069062



Arc contrasts (SUM, preview):


,book_id,A1_commitment_vows__end_minus_begin,A1_commitment_vows__middle_minus_begin,A2_emotional_safety__end_minus_begin,A2_emotional_safety__middle_minus_begin,A3_everyday_tenderness__end_minus_begin,A3_everyday_tenderness__middle_minus_begin,B1_attraction_chemistry__end_minus_begin,B1_attraction_chemistry__middle_minus_begin,B2_emotional_intimacy__end_minus_begin,...,Q_repair__end_minus_begin,Q_repair__middle_minus_begin,R1_protective_caretaking__end_minus_begin,R1_protective_caretaking__middle_minus_begin,R2_alpha_guarding__end_minus_begin,R2_alpha_guarding__middle_minus_begin,R_jealousy_possessiveness__end_minus_begin,R_jealousy_possessiveness__middle_minus_begin,S_scene_anchors__end_minus_begin,S_scene_anchors__middle_minus_begin
0,104659050,0.000785,0.003410,-0.030247,-0.008941,-0.001367,-0.004357,-0.020078,-0.021327,-0.000540,...,0.033210,0.038518,-0.013350,0.035004,-0.000772,0.034193,-0.000402,0.034744,-0.009331,-0.027454
1,11266880,0.001485,0.000570,-0.001959,-0.001782,0.017560,0.015472,0.026763,-0.009595,0.006845,...,-0.014166,0.007742,0.019292,0.005508,-0.016721,-0.016267,-0.016795,-0.016596,-0.029542,-0.029187
2,123257687,0.003312,-0.000082,0.006642,0.038741,-0.028271,-0.029371,-0.043364,0.036034,-0.028441,...,-0.010575,-0.012006,0.010630,0.037843,-0.005579,0.019447,0.000582,0.026088,-0.010802,-0.035464
3,123446478,0.000852,0.000172,-0.055483,0.001099,0.040246,-0.021483,-0.047084,-0.084547,0.032710,...,-0.082362,-0.100170,-0.056791,-0.064869,0.021757,0.025681,0.009400,0.026409,0.036661,0.025647
4,127305713,-0.005345,-0.002195,-0.012544,0.000969,-0.000724,0.021499,0.035178,0.040251,-0.027460,...,0.000909,0.028226,-0.070044,0.003590,0.035524,0.033627,0.005989,0.032954,0.039339,0.022857


## Section 6 — Export tables (for Notebook 06 and reproducibility)

In [8]:
out_dir = config.out_dir
out_dir.mkdir(parents=True, exist_ok=True)

def save_df(df: pd.DataFrame, name: str) -> None:
    if df is None:
        return
    path = out_dir / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print("✓ saved", path.name)

save_df(outputs["membership_df"], "membership_topics_long")
save_df(outputs["membership_debug_df"], "membership_debug_long")
save_df(outputs["audit_df"], "pipeline_audit")
save_df(outputs["coverage_df"], "composite_coverage")
save_df(outputs["diag_df"], "composite_diagnostics")
save_df(outputs["pca_scores_long_df"], "composite_pca_scores_long")
save_df(outputs["registry_df"], "composite_registry")
save_df(outputs["book_indices_raw"], "book_indices_raw")
save_df(outputs["book_indices_z"], "book_indices_z")
save_df(outputs["book_indices_max_raw"], "book_indices_max_raw")
save_df(outputs["book_indices_max_z"], "book_indices_max_z")
save_df(outputs["book_wide"], "book_topic_wide")

if outputs["segment_indices_raw"] is not None:
    save_df(outputs["segment_indices_raw"], "segment_indices_raw")
if outputs["segment_indices_z"] is not None:
    save_df(outputs["segment_indices_z"], "segment_indices_z")
if outputs.get("segment_indices_max_raw") is not None:
    save_df(outputs["segment_indices_max_raw"], "segment_indices_max_raw")
if outputs.get("segment_indices_max_z") is not None:
    save_df(outputs["segment_indices_max_z"], "segment_indices_max_z")
if outputs.get("arc_contrasts_sum") is not None:
    save_df(outputs["arc_contrasts_sum"], "arc_contrasts_sum")
if outputs.get("arc_contrasts_max") is not None:
    save_df(outputs["arc_contrasts_max"], "arc_contrasts_max")

print("\n✅ Export complete.")
print("Output folder:", out_dir)


✓ saved membership_topics_long.parquet
✓ saved membership_debug_long.parquet
✓ saved pipeline_audit.parquet
✓ saved composite_coverage.parquet
✓ saved composite_diagnostics.parquet
✓ saved composite_pca_scores_long.parquet
✓ saved composite_registry.parquet
✓ saved book_indices_raw.parquet
✓ saved book_indices_z.parquet
✓ saved book_indices_max_raw.parquet
✓ saved book_indices_max_z.parquet
✓ saved book_topic_wide.parquet
✓ saved segment_indices_raw.parquet
✓ saved segment_indices_z.parquet
✓ saved segment_indices_max_raw.parquet
✓ saved segment_indices_max_z.parquet
✓ saved arc_contrasts_sum.parquet
✓ saved arc_contrasts_max.parquet

✅ Export complete.
Output folder: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/measurement_v5


In [9]:
## Section 7 — Export audit tables as CSV (for manual inspection)

# Create audit subfolder in results directory
audit_dir = Path(config.out_dir) / "audit"
audit_dir.mkdir(parents=True, exist_ok=True)

# List of files to export as CSV (only the ones specified)
audit_files = [
    "composite_registry",
    "pipeline_audit",
    "membership_topics_long",
    "membership_debug_long",
    "composite_diagnostics",
    "book_indices_raw",
    "book_indices_z",
    "segment_indices_raw",
    "segment_indices_z",
    "arc_contrasts_sum",
    "arc_contrasts_max",
    "composite_pca_scores_long",
]

print("Exporting audit tables as CSV...")
for file_base in audit_files:
    parquet_path = config.out_dir / f"{file_base}.parquet"
    if parquet_path.exists():
        df = pd.read_parquet(parquet_path)
        csv_path = audit_dir / f"{file_base}.csv"
        df.to_csv(csv_path, index=False)
        print(f"✓ saved {csv_path.name} ({len(df)} rows)")
    else:
        print(f"⚠️  {file_base}.parquet not found, skipping")

print(f"\n✅ Audit CSV export complete.")
print(f"Audit folder: {audit_dir}")


Exporting audit tables as CSV...
✓ saved composite_registry.csv (26 rows)
✓ saved pipeline_audit.csv (26 rows)
✓ saved membership_topics_long.csv (477 rows)
✓ saved membership_debug_long.csv (1304 rows)
✓ saved composite_diagnostics.csv (26 rows)
✓ saved book_indices_raw.csv (92 rows)
✓ saved book_indices_z.csv (92 rows)
✓ saved segment_indices_raw.csv (276 rows)
✓ saved segment_indices_z.csv (276 rows)
✓ saved arc_contrasts_sum.csv (92 rows)
✓ saved arc_contrasts_max.csv (92 rows)
✓ saved composite_pca_scores_long.csv (2392 rows)

✅ Audit CSV export complete.
Audit folder: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/measurement_v5/audit
